# Price Prediction with TF‑IDF Weighted Word2Vec + Deep Residual MLP

This notebook combines two ideas:
1. **TF‑IDF weighted averaging** of Word2Vec word vectors – emphasises rare, informative words while keeping semantic relationships.
2. A **deep residual MLP** (similar to Wojtek’s architecture) to learn complex non‑linear interactions.

The model stays well under 100M parameters and is designed to be trained on a single GPU or CPU.

**Pipeline:**
- Load train/val/test splits via `Item.from_hub`.
- Train Word2Vec on **all** summaries (unsupervised).
- Fit a `TfidfVectorizer` on the **training** summaries to obtain IDF weights.
- For each summary, compute the TF‑IDF weighted average of its word vectors.
- Feed the resulting dense vector into a deep residual MLP.
- Train with log‑transformed prices and L1 loss.
- Evaluate on the test set.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

from gensim.models import Word2Vec

# Your Item class and evaluate function
from pricer.items import Item
from pricer.evaluator import evaluate

In [ ]:
# Configuration
LITE_MODE = False
username = "ed-donner"  
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

# Load data
train_items, val_items, test_items = Item.from_hub(dataset)
print(f"Train: {len(train_items):,} | Val: {len(val_items):,} | Test: {len(test_items):,}")

# For speed, we may subsample during development; remove these lines for full training
# train_items = random.sample(train_items, 100_000)
# val_items   = val_items[:5000]
# test_items  = test_items[:5000]

## 1. Train Word2Vec on All Summaries

We combine all summaries (train+val+test) to build a rich vocabulary. The vector size is set to 200 – a good trade‑off between expressiveness and parameter count.

In [ ]:
# Collect all summaries (handle None)
def safe_summary(item):
    return item.summary if item.summary is not None else ""

all_summaries = [safe_summary(item) for item in train_items + val_items + test_items]

# Tokenize (using nltk.word_tokenize)
print("Tokenizing all summaries...")
tokenized_all = [word_tokenize(s.lower()) for s in tqdm(all_summaries)]

# Train Word2Vec
w2v_model = Word2Vec(sentences=tokenized_all,
                     vector_size=200,          # 200-dimensional vectors
                     window=5,
                     min_count=3,              # ignore very rare words
                     workers=4,
                     epochs=10)

print(f"Vocabulary size: {len(w2v_model.wv.key_to_index)}")

## 2. Fit TF‑IDF Vectorizer on Training Summaries

We fit a `TfidfVectorizer` on the **training** summaries only, to obtain IDF weights. These weights will be used later to compute weighted averages.

We keep the same tokenisation as Word2Vec (lowercase, word_tokenize) so that terms match.

`max_features` is set to 20,000 – this is just to have a manageable vocabulary for IDF; the actual TF‑IDF scores will be applied to words that appear in the Word2Vec vocabulary.

In [ ]:
train_summaries = [safe_summary(item) for item in train_items]

# Custom tokenizer that matches Word2Vec's preprocessing
def w2v_tokenizer(text):
    return word_tokenize(text.lower())

tfidf = TfidfVectorizer(tokenizer=w2v_tokenizer,
                        lowercase=False,          # we already lowercased in tokenizer
                        max_features=20000,
                        norm=None,                # we'll handle averaging ourselves
                        use_idf=True,
                        smooth_idf=True)

# Fit on training summaries
tfidf.fit(train_summaries)
print(f"TF‑IDF vocabulary size: {len(tfidf.vocabulary_)}")

# Build a mapping from word to its IDF value (for fast lookup)
idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

## 3. Function to Compute TF‑IDF Weighted Average

For a given summary, we:
- Tokenise into words.
- For each word present in both the Word2Voc model and the TF‑IDF dictionary, retrieve its vector and IDF weight.
- Compute the weighted average: sum(word_vector * idf_weight) / sum(idf_weights).
- If no words match, return a zero vector.

In [ ]:
def tfidf_weighted_avg(summary, w2v_model, idf_dict, vector_size=200):
    words = word_tokenize(summary.lower())
    vec_sum = np.zeros(vector_size, dtype=np.float32)
    weight_sum = 0.0
    for w in words:
        if w in w2v_model.wv and w in idf_dict:
            vec_sum += w2v_model.wv[w] * idf_dict[w]
            weight_sum += idf_dict[w]
    if weight_sum > 0:
        return vec_sum / weight_sum
    else:
        return vec_sum   # all zeros

## 4. Prepare Feature Matrices for Train/Val/Test

We apply the function to every summary. This loop can be slow on 800k items!

In [ ]:
def compute_features(items, w2v_model, idf_dict, vector_size=200):
    features = []
    for it in tqdm(items, desc="Computing features"):
        summary = safe_summary(it)
        vec = tfidf_weighted_avg(summary, w2v_model, idf_dict, vector_size)
        features.append(vec)
    return np.array(features, dtype=np.float32)

print("Processing training summaries...")
X_train = compute_features(train_items, w2v_model, idf_dict)
y_train = np.array([it.price for it in train_items], dtype=np.float32)

print("Processing validation summaries...")
X_val = compute_features(val_items, w2v_model, idf_dict)
y_val = np.array([it.price for it in val_items], dtype=np.float32)

print("Processing test summaries...")
X_test = compute_features(test_items, w2v_model, idf_dict)
y_test = np.array([it.price for it in test_items], dtype=np.float32)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

## 5. Target Transformation (Log + Standardisation)

We apply a log transform to reduce skew and then standardise to zero mean and unit variance for training stability.

In [ ]:
# Log transform (adding 1 to avoid log(0))
y_train_log = np.log(y_train + 1)
y_val_log   = np.log(y_val + 1)
y_test_log  = np.log(y_test + 1)

# Compute mean and std from training set
y_mean = y_train_log.mean()
y_std  = y_train_log.std()

# Standardise
y_train_norm = (y_train_log - y_mean) / y_std
y_val_norm   = (y_val_log - y_mean) / y_std
y_test_norm  = (y_test_log - y_mean) / y_std

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_norm, dtype=torch.float32).unsqueeze(1)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
y_val_t   = torch.tensor(y_val_norm, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_norm, dtype=torch.float32).unsqueeze(1)

## 6. Define the Deep Residual MLP

Input size is 200 (the Word2Vec dimension). Hidden size is 2048, with 5 residual blocks. This yields about 42M parameters – still under 100M.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))

class DeepMLP(nn.Module):
    def __init__(self, input_size=200, hidden_size=2048, num_blocks=5, dropout=0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_size, dropout) for _ in range(num_blocks)
        ])
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x)

In [ ]:
# Device setup
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Instantiate model
model = DeepMLP(input_size=200, hidden_size=2048, num_blocks=5, dropout=0.2).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,} (under 100M: {total_params < 100_000_000})")

## 7. Prepare DataLoaders

In [ ]:
batch_size = 256   # adjust based on GPU memory
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
val_ds = TensorDataset(X_val_t, y_val_t)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

## 8. Training Loop

We use L1 loss (mean absolute error on the normalised log scale), AdamW optimizer, cosine annealing scheduler, and gradient clipping.

In [ ]:
criterion = nn.L1Loss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)

num_epochs = 10
history = {'train_loss': [], 'val_loss': [], 'val_mae': []}

for epoch in range(1, num_epochs+1):
    model.train()
    train_losses = []
    loop = tqdm(train_loader, desc=f"Epoch {epoch / num_epochs}")
    for X_batch, y_batch in loop:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        loop.set_postfix(loss=np.mean(train_losses[-50:]))  # running average
    scheduler.step()

    train_loss_epoch = np.mean(train_losses)
    history['train_loss'].append(train_loss_epoch)

    # Validation
    model.eval()
    val_losses = []
    val_preds_norm = []
    val_targets_norm = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out = model(X_batch)
            loss = criterion(out, y_batch)
            val_losses.append(loss.item())
            val_preds_norm.append(out.cpu())
            val_targets_norm.append(y_batch.cpu())
    val_loss_epoch = np.mean(val_losses)
    history['val_loss'].append(val_loss_epoch)

    # Convert back to dollar MAE
    val_preds_norm = torch.cat(val_preds_norm).numpy().flatten()
    val_targets_norm = torch.cat(val_targets_norm).numpy().flatten()
    val_preds = np.exp(val_preds_norm * y_std + y_mean) - 1
    val_targets = np.exp(val_targets_norm * y_std + y_mean) - 1
    val_mae = mean_absolute_error(val_targets, val_preds)
    history['val_mae'].append(val_mae)

    print(f"Epoch {epoch}: train loss={train_loss_epoch:.4f}, val loss={val_loss_epoch:.4f}, val MAE=${val_mae:.2f}")

## 9. Plot Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss'])+1)

fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(epochs_range, history['train_loss'], label='Train loss')
ax1.plot(epochs_range, history['val_loss'], label='Val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (L1 on normalized log price)')
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.title('Training and Validation Loss')
plt.show()

fig, ax2 = plt.subplots(figsize=(8,4))
ax2.plot(epochs_range, history['val_mae'], marker='o', color='C2')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation MAE ($)')
ax2.grid(True, alpha=0.3)
plt.title('Validation MAE')
plt.show()

## 10. Prediction Function 

This function takes an `Item` and returns a predicted price in dollars.

In [ ]:
def my_model(item):
    model.eval()
    summary = safe_summary(item)
    vec = tfidf_weighted_avg(summary, w2v_model, idf_dict, vector_size=200)
    X = torch.tensor(vec, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_norm = model(X).item()
    pred_price = np.exp(pred_norm * y_std + y_mean) - 1
    return max(0, pred_price)   # ensure non‑negative

## 11. Final Evaluation on Test Set

In [ ]:
evaluate(my_model, test_items, size=1000)

## 12. Save Model and Vectorizers

Saves the trained components for later use.

In [ ]:
torch.save(model.state_dict(), 'deep_mlp_w2v_tfidf.pth')
w2v_model.save('word2vec.model')
import joblib
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
# Save normalisation stats
np.savez('norm_stats.npz', y_mean=y_mean, y_std=y_std)